<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

This project asks whether content-page characteristics are associated with a declining content label and whether machine learning can rank pages for human review. The analysis uses 30,000 pseudonymized content items across 32 clients, with a declining-label base rate of approximately 54.2%. A Random Forest using 20 content, search, and engagement features was evaluated against a transparent baseline using five-fold client-grouped cross-validation, Precision@100, and ROC-AUC, alongside a leakage audit and error analysis. The Random Forest achieved a mean Precision@100 of 0.800 ± 0.079 and mean ROC-AUC of 0.679 ± 0.043, compared with 0.508 ± 0.207 and 0.522 ± 0.043 for the baseline, but overlapping 90-day feature and label windows mean the results should be interpreted as directional decision-support evidence rather than future-period prediction. The final output is a ranked human-review queue with reason codes and recommended investigation actions, with explicit boundaries against automatically publishing, deleting, rewriting, or making causal claims from the model.


## 1. Question

*The research question and the decision it supports.*

research question - which content page characteristics ae associated with a declining content label and can a machine learning model use those signals to rank pages for human review

Decision supported

The intended decision is which content pages should be reviewed first by a content for SEO team

The unit of analysis is a content page. The model produces a ranking and declining label probability, which is converted into a human review queue

a reviewer can then investigate the page's performance, freshness,search intent, and other available context before deciding whether any action is appropriate

the cost of a wrong call is important. Prioritizing a healthy page can waste limited editorial resources, while overlooking a page that deserves attention can delay investigation. for this reason, the model is used as decision support rather than an automatic content change system

machine learning is useful here because the dataset contains multiple content, search, and engagement signals that can be combined into a ranking rather than relying on one simple rule

## 2. Data

The analysis uses the FlyRank internship starter dataset:

`data/raw/content_refresh_anonymized.csv`

The dataset contains **30,000 pseudonymized content items across 32 clients**, with trailing 90-day search and engagement measurements.

The analysis uses content-level features including search demand, competition, content characteristics, traffic and engagement measurements, content age, update recency, and categorical content attributes.

The target is:

`is_declining_label = 1` when `trend_direction == "down"`.

The target base rate in the starter dataset is approximately **54.2%**.

### Deliberately excluded fields

The following fields were excluded from model features:

* `trend_pct`
* `trend_direction`
* `impressions_last_30d`
* `impressions_prev_30d`
* `clicks_last_30d`
* `clicks_prev_30d`
* `sessions_last_30d`
* `sessions_prev_30d`
* `is_declining_label`
* `content_id`
* `client_id`

The trend fields and last/previous-30-day fields are excluded because they are directly involved in constructing the declining label and create leakage risk.

The content and client IDs are pseudonymous identifiers. They are used for grouping, splitting, and identification where necessary, but are not model features.

The analysis also follows the dataset guidance that `avg_position = 0` represents missing position information rather than a real rank of zero, and that rate columns are stored as percentage-style values rather than proportions.

No client-identifying information is included in the public analysis.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



The analysis followed a staged workflow: problem framing, data-contract checks, signal analysis, baseline/model development, honest validation, leakage auditing, and action-playbook design.

The final model is a **Random Forest classifier** used to rank pages according to their probability of belonging to the declining-label class.

The final feature set contains 20 features:

### Numeric features

* `search_volume`
* `competition`
* `cpc`
* `word_count`
* `impressions_90d`
* `clicks_90d`
* `sessions_90d`
* `engaged_sessions_90d`
* `content_age_days`
* `days_since_last_update`
* `ctr`
* `avg_position`
* `engagement_rate`
* `scroll_rate`
* `ai_traffic_pct`
* `days_with_impressions`
* `days_with_sessions`

### Categorical features

* `content_type`
* `main_intent`
* `competition_level`

The model uses median imputation and scaling for numeric variables and most-frequent imputation with one-hot encoding for categorical variables.

The Random Forest uses 300 trees, `min_samples_leaf=20`, `n_jobs=-1`, and `random_state=42`.

### Label definition

The target represents whether the content item has the declining label defined by the supplied `trend_direction` field.

### Validation

The main honest evaluation groups observations by `client_id` using GroupKFold so that pages from the same client do not appear in both training and validation folds.

This tests whether the model's ranking signal transfers across clients rather than benefiting from seeing pages from the same client during training.

### Leakage checks

The analysis explicitly excludes label-derived fields, sibling label fields, pseudonymous IDs as predictive features, and decision-derived flags.

A remaining limitation is that several 90-day features overlap the period used to construct the declining label. Client-grouped validation does not remove this same-window temporal overlap, so the resulting evidence is treated as directional decision support rather than deployment-ready future prediction.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# ============================================
# CAPSTONE — RESULTS VS BASELINE
# Fresh reproducible evaluation
# ============================================

import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 42

# --------------------------------------------
# 1. Load the starter dataset
# --------------------------------------------

URL = (
    "https://raw.githubusercontent.com/"
    "ritakimani9-lang/machinelearning/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(URL)

# Recreate the target exactly as in W05
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Target base rate:", round(df["is_declining_label"].mean(), 3))


# --------------------------------------------
# 2. Recreate the W05 baseline exactly
# --------------------------------------------

stale = df["freshness_tier"].isin(["91-180", "181+"])

visible = df["impression_tier"].isin(
    ["moderate", "good", "excellent"]
)

df["baseline_score"] = (
    stale & visible
) * df["impressions_90d"]


# --------------------------------------------
# 3. Define the exact W05 model features
# --------------------------------------------

NUM = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions",
]

CAT = [
    "content_type",
    "main_intent",
    "competition_level",
]

LEAKY = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "is_declining_label",
    "content_id",
    "client_id",
]

assert not set(NUM + CAT) & set(LEAKY), \
    "Leakage detected in model features."


# --------------------------------------------
# 4. Preprocessing
# --------------------------------------------

def prep(scale):
    return ColumnTransformer([
        (
            "numeric",
            make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler() if scale else "passthrough"
            ),
            NUM
        ),
        (
            "categorical",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(handle_unknown="ignore")
            ),
            CAT
        )
    ])


# --------------------------------------------
# 5. Models
# --------------------------------------------

models = {
    "Logistic Regression": make_pipeline(
        prep(True),
        LogisticRegression(max_iter=2000)
    ),

    "Random Forest": make_pipeline(
        prep(False),
        RandomForestClassifier(
            300,
            min_samples_leaf=20,
            n_jobs=-1,
            random_state=SEED
        )
    )
}


# --------------------------------------------
# 6. Precision@K
# --------------------------------------------

def precision_at_k(scores, y_true, k):
    order = np.argsort(
        -scores,
        kind="stable"
    )[:k]

    return y_true[order].mean()


# --------------------------------------------
# 7. Prepare data
# --------------------------------------------

X = df[NUM + CAT]

y = df["is_declining_label"].values

groups = df["client_id"].values

K = 100

cv = GroupKFold(n_splits=5)

results = []


# --------------------------------------------
# 8. Client-grouped evaluation
# --------------------------------------------

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups)
):

    y_test = y[test_idx]

    fold_result = {
        "fold": fold,
        "base_rate": y_test.mean(),
        "test_clients": len(
            set(groups[test_idx])
        )
    }

    # ---- Baseline ----

    baseline_scores = (
        df["baseline_score"]
        .values[test_idx]
    )

    fold_result["Baseline_Precision@100"] = (
        precision_at_k(
            baseline_scores,
            y_test,
            K
        )
    )

    fold_result["Baseline_ROC_AUC"] = (
        roc_auc_score(
            y_test,
            baseline_scores
        )
    )


    # ---- Machine-learning models ----

    for name, model in models.items():

        model.fit(
            X.iloc[train_idx],
            y[train_idx]
        )

        probabilities = model.predict_proba(
            X.iloc[test_idx]
        )[:, 1]

        fold_result[
            f"{name}_Precision@100"
        ] = precision_at_k(
            probabilities,
            y_test,
            K
        )

        fold_result[
            f"{name}_ROC_AUC"
        ] = roc_auc_score(
            y_test,
            probabilities
        )

    results.append(fold_result)


# --------------------------------------------
# 9. Results table
# --------------------------------------------

results_df = pd.DataFrame(results)

print("\nFOLD RESULTS")
display(results_df.round(3))


# --------------------------------------------
# 10. Mean ± standard deviation
# --------------------------------------------

summary = pd.DataFrame({
    "Evaluation": [
        "Baseline",
        "Logistic Regression",
        "Random Forest"
    ],

    "Precision@100": [
        results_df["Baseline_Precision@100"].mean(),
        results_df["Logistic Regression_Precision@100"].mean(),
        results_df["Random Forest_Precision@100"].mean()
    ],

    "Precision@100_std": [
        results_df["Baseline_Precision@100"].std(),
        results_df["Logistic Regression_Precision@100"].std(),
        results_df["Random Forest_Precision@100"].std()
    ],

    "ROC-AUC": [
        results_df["Baseline_ROC_AUC"].mean(),
        results_df["Logistic Regression_ROC_AUC"].mean(),
        results_df["Random Forest_ROC_AUC"].mean()
    ],

    "ROC-AUC_std": [
        results_df["Baseline_ROC_AUC"].std(),
        results_df["Logistic Regression_ROC_AUC"].std(),
        results_df["Random Forest_ROC_AUC"].std()
    ],

    "Base rate": [
        results_df["base_rate"].mean(),
        results_df["base_rate"].mean(),
        results_df["base_rate"].mean()
    ]
})

print("\nFINAL GROUPED VALIDATION SUMMARY")
display(summary.round(3))

Rows: 30000
Clients: 32
Target base rate: 0.542

FOLD RESULTS


,fold,base_rate,test_clients,Baseline_Precision@100,Baseline_ROC_AUC,Logistic Regression_Precision@100,Logistic Regression_ROC_AUC,Random Forest_Precision@100,Random Forest_ROC_AUC
0,0,0.490,1,0.34,0.539,0.72,0.604,0.84,0.648
1,1,0.645,7,0.51,0.473,0.67,0.590,0.86,0.632
2,2,0.379,8,0.27,0.521,0.79,0.650,0.85,0.744
3,3,0.622,8,0.76,0.585,0.73,0.652,0.67,0.683
4,4,0.585,8,0.66,0.494,0.82,0.652,0.78,0.688



FINAL GROUPED VALIDATION SUMMARY


,Evaluation,Precision@100,Precision@100_std,ROC-AUC,ROC-AUC_std,Base rate
0,Baseline,0.508,0.207,0.522,0.043,0.544
1,Logistic Regression,0.746,0.059,0.629,0.030,0.544
2,Random Forest,0.800,0.079,0.679,0.043,0.544


## 4. Results (vs baseline)

The model was evaluated using 5-fold client-grouped cross-validation. The primary ranking metric is Precision@100, reported alongside ROC-AUC and the validation base rate.

The baseline, Logistic Regression, and Random Forest were evaluated on the same grouped validation folds.

| **Evaluation**      | **Precision@100** |       **ROC-AUC** | **Base rate** |
| ------------------- | ----------------: | ----------------: | ------------: |
| Baseline            |     0.508 ± 0.207 |     0.522 ± 0.043 |         0.544 |
| Logistic Regression |     0.746 ± 0.059 |     0.629 ± 0.030 |         0.544 |
| Random Forest       | **0.800 ± 0.079** | **0.679 ± 0.043** |         0.544 |

The Random Forest produced the strongest measured ranking performance in this evaluation. Its mean Precision@100 was 0.800 compared with 0.508 for the baseline, an absolute difference of 0.292.

The Random Forest also had the highest mean ROC-AUC at 0.679, compared with 0.522 for the baseline. These results indicate measured directional signal for ranking pages associated with the declining label.

However, performance varied across client-grouped folds. Random Forest Precision@100 ranged from 0.67 to 0.86, while fold base rates also varied. The results should therefore be interpreted as evidence of directional ranking signal in this dataset rather than as proof of universal or future-period performance.

The Logistic Regression model also outperformed the baseline, with a mean Precision@100 of 0.746 and mean ROC-AUC of 0.629. This provides evidence that the observed signal is not limited to the Random Forest model alone.

The grouped evaluation is more relevant to the intended unseen-client generalization question than the earlier random-split evaluation because pages from the same client are kept within the same validation fold.

### Error analysis

False positives and false negatives were reviewed to identify cases where the model's ranking signal did not correspond cleanly to the observed declining label.

These errors reinforce that the model should prioritize investigation rather than automatically determine a content action. A high model score identifies a page that may be worth reviewing; it does not establish why the page is declining or which intervention will improve its future performance.

The final numerical values above were generated from a fresh execution of the capstone evaluation and are therefore traceable to the reproducible evaluation code.


### Error analysis

A qualitative inspection of the model-generated ranking showed that the highest-ranked pages were strongly concentrated in the observed declining class. All of the top 20 ranked pages in the full-data diagnostic ranking had the declining label, with predicted decline probabilities ranging from 0.953 to 0.974.

The highest-ranked pages were predominantly keyword articles and were generally informational. Several of the top-ranked pages were approximately 132–147 days old and had been updated 20 days earlier. This suggests that the ranking is responding to combinations of content, search, and engagement signals rather than simply assigning high priority to the oldest pages.

The diagnostic inspection also showed that some high-scoring pages did not have the declining label. These cases demonstrate why the model output should be treated as a prioritization signal rather than a definitive content diagnosis.

The error inspection was performed as a qualitative diagnostic on the full dataset and is not used as a substitute for the client-grouped cross-validation metrics reported above. Formal performance claims are based on the held-out grouped evaluation.

This distinction is important because a model fitted on the full dataset can be useful for inspecting ranking patterns, but its in-sample classifications should not be interpreted as an unbiased estimate of generalization performance.


In [ ]:
# ============================================
# CAPSTONE — ERROR ANALYSIS
# ============================================

# Fit the final Random Forest on the full dataset
# This is for inspecting ranked cases, NOT for
# reporting a new performance estimate.

final_rf = make_pipeline(
    prep(False),
    RandomForestClassifier(
        300,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=SEED
    )
)

final_rf.fit(X, y)

# Generate decline probabilities
df["decline_probability"] = final_rf.predict_proba(X)[:, 1]

# Rank all pages
df["priority_rank"] = (
    df["decline_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# --------------------------------------------
# Top-ranked pages
# --------------------------------------------

top_ranked = (
    df.sort_values(
        "decline_probability",
        ascending=False
    )
    [
        [
            "priority_rank",
            "decline_probability",
            "is_declining_label",
            "content_type",
            "main_intent",
            "content_age_days",
            "days_since_last_update",
            "avg_position",
            "impressions_90d",
            "word_count"
        ]
    ]
    .head(20)
)

print("TOP 20 RANKED PAGES")
display(top_ranked.round(3))


# --------------------------------------------
# False-positive / false-negative style review
# --------------------------------------------

# Define a practical review threshold for inspection.
# This threshold is NOT a deployment threshold.
REVIEW_THRESHOLD = 0.75

df["review_prediction"] = (
    df["decline_probability"] >= REVIEW_THRESHOLD
).astype(int)

false_positives = df[
    (df["review_prediction"] == 1) &
    (df["is_declining_label"] == 0)
].copy()

false_negatives = df[
    (df["review_prediction"] == 0) &
    (df["is_declining_label"] == 1)
].copy()

print("False-positive-style cases:", len(false_positives))
print("False-negative-style cases:", len(false_negatives))


# --------------------------------------------
# Summarise the two error groups
# --------------------------------------------

error_summary = pd.DataFrame({
    "Group": [
        "False-positive-style",
        "False-negative-style"
    ],

    "Rows": [
        len(false_positives),
        len(false_negatives)
    ],

    "Mean content age": [
        false_positives["content_age_days"].mean(),
        false_negatives["content_age_days"].mean()
    ],

    "Mean avg position": [
        false_positives["avg_position"].mean(),
        false_negatives["avg_position"].mean()
    ],

    "Mean impressions": [
        false_positives["impressions_90d"].mean(),
        false_negatives["impressions_90d"].mean()
    ],

    "Mean word count": [
        false_positives["word_count"].mean(),
        false_negatives["word_count"].mean()
    ]
})

print("\nERROR GROUP SUMMARY")
display(error_summary.round(2))


# --------------------------------------------
# Content-type profile
# --------------------------------------------

fp_content_type = (
    false_positives["content_type"]
    .value_counts()
    .rename("count")
    .to_frame()
)

fn_content_type = (
    false_negatives["content_type"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nFALSE-POSITIVE-STYLE CASES BY CONTENT TYPE")
display(fp_content_type)

print("\nFALSE-NEGATIVE-STYLE CASES BY CONTENT TYPE")
display(fn_content_type)

TOP 20 RANKED PAGES


,priority_rank,decline_probability,is_declining_label,content_type,main_intent,content_age_days,days_since_last_update,avg_position,impressions_90d,word_count
2063,1,0.974,1,keyword article,informational,147,20,26.4,9710,4574.0
7478,2,0.968,1,keyword article,informational,147,20,24.1,5566,3990.0
12809,3,0.965,1,keyword article,informational,141,20,23.7,4434,3733.0
29353,4,0.963,1,keyword article,informational,141,20,25.5,4282,3829.0
21344,5,0.963,1,keyword article,informational,147,20,26.8,3288,3862.0
15272,6,0.961,1,keyword article,informational,141,20,28.0,10171,3817.0
24008,7,0.960,1,keyword article,informational,147,20,29.2,4615,3729.0
2452,8,0.959,1,keyword article,informational,141,20,29.8,7471,3897.0
19347,9,0.958,1,keyword article,informational,141,20,31.2,22184,4289.0
18891,10,0.958,1,keyword article,informational,147,20,22.0,8156,3743.0


False-positive-style cases: 256
False-negative-style cases: 11064

ERROR GROUP SUMMARY


,Group,Rows,Mean content age,Mean avg position,Mean impressions,Mean word count
0,False-positive-style,256,164.96,14.92,3351.84,3438.91
1,False-negative-style,11064,263.89,16.21,5564.80,3198.72



FALSE-POSITIVE-STYLE CASES BY CONTENT TYPE


,count
content_type,
keyword article,255
feedly article,1



FALSE-NEGATIVE-STYLE CASES BY CONTENT TYPE


,count
content_type,
keyword article,10177
feedly article,504
comparison article,383


## 5. Limitations

*What this work cannot claim.*



This analysis has several important limitations.

First, the starter dataset contains 30,000 content items across 32 clients. The results therefore describe this dataset and its observed population rather than all websites or all future content.

Second, the honest evaluation groups pages by client, but it does not establish future-period performance. A time-aware evaluation using strictly pre-outcome features would provide stronger evidence for future deployment.

Third, several 90-day features overlap the period used to construct the declining label. Although direct label-derived fields were excluded, this same-window overlap remains a temporal leakage concern.

Fourth, the model is evaluated as a ranking system rather than as a causal intervention. A high model score indicates that a page is worth reviewing; it does not establish that changing the page will improve its future performance.

Finally, the model should not be interpreted as predicting or explaining Google's ranking algorithm. It models an outcome defined within the supplied FlyRank dataset.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The model output is converted into a ranked review queue for human investigation. The queue is designed to answer a practical question: **which pages should a content team spend review time on first?**

The W07 action playbook assigns each ranked page a reason code and a suggested human action.

| **Reason code**      | **Meaning**                                              | **Suggested human action**                                |
| -------------------- | -------------------------------------------------------- | --------------------------------------------------------- |
| `AGE_AND_DECLINE`    | Decline signal combined with relatively high content age | Review freshness, search performance, and content quality |
| `DECLINE_RISK`       | Strong model signal without the additional age condition | Review page performance and investigate possible causes   |
| `PERFORMANCE_REVIEW` | Moderate model signal                                    | Inspect search and engagement signals before acting       |
| `MONITOR`            | Weaker model signal                                      | Monitor rather than prioritize for immediate change       |

A FlyRank editor could use the queue to decide **where to spend review time first**, then investigate the page using additional context that is not represented in the model score.

The recommended workflow is:

**Model ranking → reason code → human investigation → editorial decision → monitoring**

The model should not automatically publish, delete, rewrite, change search intent, or make other business-critical content decisions.

The recommendations are therefore best described as **directional decision support**. The queue identifies pages that appear worth reviewing first; it does not establish the cause of decline or guarantee that a particular intervention will improve future performance.

The ranked queue is reproducible from the W07 notebook. The generated CSV is intentionally kept out of version control, while the notebook preserves the process required to regenerate it.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The paper reuses the reproducible outputs produced during the internship rather than manually recreating results.

The key artifacts are:

* **Model evaluation metrics:** the grouped-validation results from W05, including Precision@100, ROC-AUC, and validation base rate.
* **Baseline comparison:** the transparent baseline evaluated on the same client-grouped folds as the machine-learning models.
* **Validation comparison:** the W05/W06 comparison between random-split and client-grouped evaluation.
* **Leakage audit:** the W06 review of label-derived fields, identifiers, decision-derived variables, and the remaining same-window temporal overlap.
* **Error analysis:** the capstone diagnostic inspection of highly ranked pages and the characteristics of cases that do not align cleanly with the observed label.
* **Action playbook:** the W07 ranked review queue and its reason-code/action mapping.
* **Observed-pattern figures:** figures showing relevant dataset patterns that support the interpretation and recommendations.

The W07 ranked queue is regenerated by the notebook and written to:

`work/outputs/w07_ranked_action_queue.csv`

The queue is intentionally not committed as a data file. The notebook preserves the reproducible process required to regenerate it.

Public-safe figures and summary metrics can be committed for reuse by the deployed paper, while client-identifying information, private queries, and raw operational data remain excluded.


## 8. Reproducibility

The analysis is designed so that the main results can be regenerated from the repository and the supplied FlyRank internship dataset.

### Data

The starter dataset is:

`data/raw/content_refresh_anonymized.csv`

The capstone evaluation uses the 30,000-row starter dataset and recreates the target from:

`is_declining_label = (trend_direction == "down")`

### Model

The final Random Forest configuration is:

* `n_estimators = 300`
* `min_samples_leaf = 20`
* `random_state = 42`
* `n_jobs = -1`

The model uses the documented 20-feature set with median imputation for numeric features and most-frequent imputation plus one-hot encoding for categorical features.

### Validation

The primary evaluation uses:

* 5-fold `GroupKFold`
* grouping variable: `client_id`
* primary ranking metric: Precision@100
* secondary metric: ROC-AUC
* validation base rate reported alongside the ranking metrics

The grouped evaluation is intended to reduce client-level overlap between training and validation folds. It does not establish future-period performance.

### Re-running the analysis

The notebooks should be executed from top to bottom in order:

1. `w01_research_question.ipynb`
2. `w02_ml_task_framing.ipynb`
3. `w03_data_contract.ipynb`
4. `w04_signal_audit.ipynb`
5. `w05_model.ipynb`
6. `w06_validation_audit.ipynb`
7. `w07_action_playbook.ipynb`
8. `capstone.ipynb`

The capstone evaluation cell reproduces the baseline and model comparison using the same grouped-validation design and fixed random seed.

The W07 action queue can be regenerated from its notebook. The generated queue is not required to be committed to version control because the reproducible generation process is preserved in the notebook.

### Reproducibility boundary

The analysis does not claim a sealed future-period holdout evaluation. Several 90-day features overlap the period used to construct the declining label, so the reported model results should be interpreted as directional decision-support evidence rather than as proof of future deployment performance.


## 5-Minute Demo Outline

**0:00–0:45 — Problem and decision**

* Explain the research question: which content pages show signals associated with decline?
* Explain the intended decision: which pages should a content/SEO team review first?
* Emphasize that the system provides decision support, not automatic content changes.

**0:45–1:30 — Data and methodology**

* 30,000 pseudonymized content items.
* 32 clients.
* 54.2% declining-label base rate.
* 20 model features covering content, search, and engagement signals.
* Random Forest compared with a simple baseline.
* Client-grouped cross-validation used to test generalization across clients.

**1:30–2:30 — Validation and results**

* Show the baseline versus Logistic Regression versus Random Forest results.
* Highlight Random Forest Precision@100 = 0.800 ± 0.079.
* Highlight Random Forest ROC-AUC = 0.679 ± 0.043.
* Explain why client grouping makes the evaluation more demanding than a simple random split.

**2:30–3:30 — What the model produces**

* Show the ranked review queue.
* Explain the reason codes.
* Demonstrate how a high-ranked page moves from model score → reason code → human investigation.
* Connect the model output to the W07 action playbook.

**3:30–4:30 — Error analysis and limitations**

* Explain that the error analysis was diagnostic rather than a formal held-out confusion matrix.
* Discuss the 90-day feature/label overlap.
* Explain why the current evaluation does not establish future-period prediction or deployment readiness.
* Reinforce that model associations should not be interpreted as causal explanations.

**4:30–5:00 — Final decision-support workflow**

**Model ranking → Reason code → Human investigation → Editorial decision → Monitoring**

Close by explaining that the main deliverable is not simply a model score, but a reproducible workflow that connects ML evidence to a controlled human decision process.


## 9. Acknowledgments & data credit

This project was completed as part of the FlyRank Machine Learning Internship and was built on the FlyRank ML Internship dataset.

Data and internship context are credited to FlyRank:

https://flyrank.ai

The analysis uses the supplied pseudonymized dataset and does not disclose client-identifying information, private queries, or other confidential operational details.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
